In [1]:
pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 259.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 334.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 262.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 259.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 254.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 250.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 277.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 253.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

### 1. Configure for 16-bit LoRA (fp16) and Import Libraries

We'll set `load_in_4bit = False` and specify `torch.bfloat16` for 16-bit training. If your GPU does not support bfloat16, you can use `torch.float16` instead.

In [9]:
import torch
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

# Check if bfloat16 is supported, otherwise use float16
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    dtype = torch.bfloat16
    print("Using bfloat16 (recommended for Ampere GPUs and newer)")
else:
    dtype = torch.float16
    print("Using float16 (for older GPUs or if bfloat16 is not supported)")

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen1.5-0.5B-Chat", # Changed to a standard Qwen 1.5 base model
    max_seq_length = 2048,
    dtype = dtype, # Specify 16-bit dtype
    load_in_4bit = False, # IMPORTANT: Set to False for 16-bit LoRA (not QLoRA)
)

# Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA attention dimension
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Corrected to "unsloth"
    random_state = 3407,
    use_rslora = False,
    loftq_config = None, # LoftQ is for QLoRA
)

Using float16 (for older GPUs or if bfloat16 is not supported)
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.4.8 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


### 2. Prepare Dummy Dataset for 5 Tasks

We'll create a simple dummy dataset with 5 different conversation examples to simulate 5 tasks. Each conversation will follow the Llama-3 instruction format.

In [10]:
from datasets import Dataset

alpaca_prompt = """
Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Dummy conversations for 5 tasks
dummy_conversations = [
    {
        "instruction": "Tell me a fun fact about pandas.",
        "input": "",
        "response": "Giant pandas eat bamboo for about 12 hours a day!"
    },
    {
        "instruction": "What is the capital of France?",
        "input": "",
        "response": "The capital of France is Paris."
    },
    {
        "instruction": "Describe a sunny day.",
        "input": "",
        "response": "A sunny day is bright and warm, with a clear blue sky and gentle breezes."
    },
    {
        "instruction": "What's your favorite color?",
        "input": "",
        "response": "As an AI, I don't have a favorite color, but I appreciate all shades of the rainbow!"
    },
    {
        "instruction": "Summarize the plot of Little Red Riding Hood.",
        "input": "",
        "response": "Little Red Riding Hood is a tale about a young girl who encounters a wolf on her way to visit her sick grandmother. The wolf tricks her, eats her grandmother, and then tries to eat Red Riding Hood, but they are saved by a woodsman."
    },
]

# Format conversations for training
dataset_data = []
for conv in dummy_conversations:
    formatted_text = alpaca_prompt.format(
        conv["instruction"],
        conv["input"] if conv["input"] else "",
        conv["response"]
    )
    dataset_data.append({"text": formatted_text})

dummy_dataset = Dataset.from_list(dataset_data)

print("Dummy dataset created with 5 tasks:")
for i, example in enumerate(dummy_dataset):
    print(f"--- Task {i+1} ---")
    print(example["text"])
    print("\n")

Dummy dataset created with 5 tasks:
--- Task 1 ---

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
Tell me a fun fact about pandas.

### Input:


### Response:
Giant pandas eat bamboo for about 12 hours a day!


--- Task 2 ---

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Input:


### Response:
The capital of France is Paris.


--- Task 3 ---

Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
Describe a sunny day.

### Input:


### Response:
A sunny day is bright and warm, with a clear blue sky and gentle breezes.


--- Task 4 ---

Below is an instruction that describes a

### 3. Set up and Run SFTTrainer

We'll use `unsloth`'s `SFTTrainer` for efficient fine-tuning. For a quick dummy run, we'll set `max_steps` to a small number.

In [11]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dummy_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can set to True for faster training with many short examples
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5, # Small warmup for dummy run
        max_steps = 20, # Very small number of steps for a quick dummy run
        learning_rate = 2e-4,
        fp16 = (dtype == torch.float16), # Use fp16 if dtype is float16
        bf16 = (dtype == torch.bfloat16), # Use bf16 if dtype is bfloat16
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
    ),
)

# Start training
print("Starting dummy LoRA fine-tuning...")
trainer.train()
print("Dummy LoRA fine-tuning complete!")

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting dummy LoRA fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 20 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 7,569,408 of 471,557,120 (1.61% trained)


Step,Training Loss
1,5.624944
2,5.624425
3,5.310790
4,4.700347
5,4.146049
6,3.637195
7,3.290060
8,3.004656
9,2.719987
10,2.443199


Dummy LoRA fine-tuning complete!


### 4. (Optional) Inference After Training

Once training is complete, you can test the fine-tuned model.

In [12]:
FastLanguageModel.for_inference(model)

# Example inference
instruction = "Tell me about the biggest city in the world by population."
inputs = ""
alpaca_prompt_formatted = alpaca_prompt.format(instruction, inputs, "")

outputs = tokenizer(
    alpaca_prompt_formatted,
    return_tensors = "pt"
).to("cuda")

model.eval()
with torch.no_grad():
    generated_ids = model.generate(**outputs, max_new_tokens=64, use_cache=True)
    decoded_output = tokenizer.batch_decode(generated_ids[:, outputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

print("\n--- Inference Example ---")
print(f"Instruction: {instruction}")
print(f"Generated Response: {decoded_output}")

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



--- Inference Example ---
Instruction: Tell me about the biggest city in the world by population.
Generated Response: The largest city in the world by population is New York, with over 14 million people who live in the capital of the United States. It also has a global impact on transportation and transportation, with major connections to important destinations like London, Paris,上海, and New York.

### Input:
The second most populous


### 5. Push LoRA Adapter to Hugging Face (Optional)

To push your trained LoRA adapter to Hugging Face, you'll need an authenticated Hugging Face token with write access.

1.  Go to [Hugging Face Settings](https://huggingface.co/settings/tokens).
2.  Generate a new token with 'write' role.
3.  In Colab, add the token to the secrets manager under the '🔑' in the left panel. Give it the name `HF_TOKEN`.

In [14]:
from huggingface_hub import HfApi
from google.colab import userdata

# Get Hugging Face token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Initialize the Hugging Face API
api = HfApi()

# Define your Hugging Face repo ID
# Replace 'your-username/my-qwen-0.5b-lora' with your desired repository name
repo_id = "Natnaela/my-qwen-0.5b-lora"

# Check if the repository exists, if not, create it
try:
    api.repo_info(repo_id=repo_id, token=hf_token, repo_type="model")
    print(f"Repository {repo_id} already exists.")
except Exception:
    print(f"Repository {repo_id} not found, creating it...")
    api.create_repo(repo_id=repo_id, token=hf_token, repo_type="model", private=False, exist_ok=True)
    print(f"Repository {repo_id} created successfully.")

# Save the LoRA adapter locally
model.save_pretrained("my_qwen_lora_adapter")

# Push the adapter to Hugging Face Hub
api.upload_folder(
    folder_path="my_qwen_lora_adapter",
    repo_id=repo_id,
    repo_type="model",
    token=hf_token,
)

print(f"Successfully pushed LoRA adapter to https://huggingface.co/{repo_id}")

Repository Natnaela/my-qwen-0.5b-lora already exists.
Successfully pushed LoRA adapter to https://huggingface.co/Natnaela/my-qwen-0.5b-lora
